# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [52]:
# Write your code below.
%load_ext dotenv
%dotenv


The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [53]:
import os
from glob import glob
import pandas as pd
# Load the environment variable
PRICE_DATA = os.getenv("PRICE_DATA")

# Find all parquet files
parquet_files = glob(os.path.join(PRICE_DATA, "**/*.parquet"), recursive=True)

For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [54]:
import dask.dataframe as dd

In [55]:
dd_px = dd.read_parquet(parquet_files).set_index("ticker")

In [56]:
dd_px

,Date,Open,High,Low,Close,Adj Close,Volume,source,Year
npartitions=60,,,,,,,,,
A,datetime64[ns],float64,float64,float64,float64,float64,float64,string,int32
ACB,...,...,...,...,...,...,...,...,...
...,...,...,...,...,...,...,...,...,...
ZEUS,...,...,...,...,...,...,...,...,...
ZEUS,...,...,...,...,...,...,...,...,...


In [79]:
dd_feat = (
    dd_px
    .groupby("ticker", group_keys=False)
    .apply(
        lambda x: (
            x.sort_values("Date", ascending=True)
             .assign(
                 Close_lag_1=lambda x: x["Close"].shift(1),
                 Adj_Close_lag_1=lambda x: x["Adj Close"].shift(1),
                 hi_lo_range=lambda x: x["High"] - x["Low"],
             )
        ),
        meta=pd.DataFrame(
            data={
                'Date': 'datetime64[ns]',
                'Open': 'f8',
                'High': 'f8',
                'Low': 'f8',
                'Close': 'f8',
                'Adj Close': 'f8',
                'Volume': 'f8',
                'source': 'object',
                'Year': 'int32',
                'Close_lag_1': 'f8',
                'Adj_Close_lag_1': 'f8',
                'hi_lo_range': 'f8'
            },
            index = pd.Index([], dtype=pd.StringDtype(), name='ticker')
        )
    )
    .assign(returns = lambda x: (x ['Close'] / x['Close_lag_1']) - 1)
)

+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [80]:
# Write your code below.
df_feat = dd_feat.compute()

In [81]:
df_feat = df_feat.reset_index()

In [84]:
df_feat = df_feat.sort_values(['ticker', 'Date'])
df_feat['returns_ma_10'] = df_feat.groupby('ticker')['returns'].transform(
    lambda x: x.rolling(10).mean()
)


Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
+ Would it have been better to do it in Dask? Why?

(1 pt)

Was it necessary to convert to pandas?
No, not strictly necessary. Dask does support rolling window calculations. However, rolling 
operations with groupby in Dask are more complicated and can produce unexpected results.
Pandas makes it simpler and more straightforward for this task.

Would it have been better to do it in Dask?

No, pandas was the better choice here for two reasons: 1) The data is small. We only have 60 stock files. This easily fits in memory. Dask is designed for datasets too big to fit in RAM - that's not our situation. ANd 2) Dask adds overhead. Dask splits data into partitions and coordinates parallel processing.For small data, this extra coordination actually makes things slower, not faster.

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.